# PCB Defect Detection — YOLOv8m Training on Kaggle (GPU Acceleration)

This notebook trains **YOLOv8 Medium (~25.9 Million parameters)** for **120 epochs** at ultra-high resolution **1248×1248** on the **PKU-Market-PCB dataset** (6 defect classes: `missing_hole`, `mouse_bite`, `open_circuit`, `short`, `spur`, `spurious_copper`).

### Kaggle Setup Instructions:
1. **Enable GPU**: In the right sidebar under **Settings** ➔ **Accelerator** ➔ select **GPU T4 x2** or **GPU P100**.
2. **Enable Internet**: In the right sidebar under **Settings** ➔ toggle **Internet ON** (required to download packages and dataset).
3. **Run All Cells**: Click **Run All** (or run cell by cell).

---

### Step 1: Verify Kaggle GPU Accelerator

In [ ]:
!nvidia-smi

### Step 2: Install Ultralytics and Roboflow

In [ ]:
!pip install -q ultralytics roboflow

### Step 3: Download PCB Dataset from Roboflow

In [ ]:
import os
from roboflow import Roboflow

# Download dataset directly into Kaggle working directory
rf = Roboflow(api_key="Txn9rCczL5tK1n0PYkpx")
project = rf.workspace("jr-mqqnk").project("pcb-defects-detection-anddl")
dataset = project.version(1).download("yolov8")

data_yaml_path = os.path.join(dataset.location, "data.yaml")
print(f"\n[OK] Dataset ready at: {dataset.location}")
print(f"[OK] YAML config: {data_yaml_path}")

### Step 4: Train YOLOv8m (25.9M Parameters, 120 Epochs, 1248px Resolution)

In [ ]:
import os
from ultralytics import YOLO

# Disable WandB interactive login prompt
os.environ["WANDB_MODE"] = "disabled"

# Load pretrained YOLOv8 Medium model (~25.9M parameters)
model = YOLO("yolov8m.pt")

# Train for 120 epochs at 1248x1248 (multiple of 32 for ~1240px)
results = model.train(
    data=data_yaml_path,
    epochs=120,
    batch=8,                    # batch=8 fits safely in 16GB GPU VRAM at ~1248px
    imgsz=1248,                 # 1248px preserves hairline spurs and micro pinholes
    patience=25,                # Early stopping patience if validation plateaus
    device=0,                   # Primary Kaggle GPU
    workers=4,
    seed=42,
    cos_lr=True,                # Smooth Cosine Annealing learning rate schedule
    project="runs",
    name="pcb_yolov8m_1248px",
    exist_ok=True,
    pretrained=True,
    amp=True,                   # Automatic Mixed Precision for speed and VRAM efficiency
    fliplr=0.5,
    degrees=10.0,
    scale=0.3,
    mosaic=1.0,
    close_mosaic=15,            # Turn off mosaic for final 15 epochs to stabilize boundaries
)

print("\n[OK] Training completed successfully!")

### Step 5: Evaluate on Held-Out Test Split (140 Images at 1248px)

In [ ]:
# Load the best converged checkpoint
best_weights = "runs/pcb_yolov8m_1248px/weights/best.pt"
model_eval = YOLO(best_weights)

# Evaluate strictly on the held-out test split
test_metrics = model_eval.val(
    data=data_yaml_path,
    split="test",
    imgsz=1248,
    batch=8,
    device=0,
    plots=True
)

# Overall Summary
print("\n" + "=" * 65)
print("--- YOLOv8m (25.9M Params | 1248x1248) TEST METRICS ---")
print("=" * 65)
print(f"Test Set mAP@0.5:      {test_metrics.box.map50 * 100:.2f}%  ({test_metrics.box.map50:.4f})")
print(f"Test Set mAP@0.5:0.95: {test_metrics.box.map * 100:.2f}%  ({test_metrics.box.map:.4f})")
print(f"Mean Precision:        {test_metrics.box.mp * 100:.2f}%  ({test_metrics.box.mp:.4f})")
print(f"Mean Recall:           {test_metrics.box.mr * 100:.2f}%  ({test_metrics.box.mr:.4f})")

# F1 Score
mp, mr = test_metrics.box.mp, test_metrics.box.mr
f1 = 2 * (mp * mr) / (mp + mr + 1e-8)
print(f"Overall F1 Score:      {f1 * 100:.2f}%  ({f1:.4f})")
print("=" * 65)

# Per-Class Breakdown across all 6 PCB Defect Classes
print("\n--- PER-CLASS ACCURACY BREAKDOWN ---")
print(f"{'Class Name':<20} | {'mAP@0.5':<10} | {'Precision':<10} | {'Recall':<10}")
print("-" * 65)

for idx, class_name in enumerate(test_metrics.names.values()):
    cls_map = test_metrics.box.maps[idx] if len(test_metrics.box.maps) > idx else 0.0
    cls_p = test_metrics.box.p[idx] if len(test_metrics.box.p) > idx else 0.0
    cls_r = test_metrics.box.r[idx] if len(test_metrics.box.r) > idx else 0.0
    print(f"{class_name:<20} | {cls_map * 100:6.2f}%   | {cls_p * 100:6.2f}%   | {cls_r * 100:6.2f}%")

print("=" * 65)

### Step 6: Package Weights for 1-Click Download
On Kaggle, any files created in the working directory appear in the **Output** tab on the right sidebar for direct download.

In [ ]:
import shutil
from pathlib import Path

# Package the entire run folder (weights, curves, confusion matrix)
archive_name = "pcb_yolov8m_1248px_final"
shutil.make_archive(archive_name, "zip", "runs/pcb_yolov8m_1248px")
print(f"\n[SUCCESS] Packaged outputs into: {archive_name}.zip")
print("You can download this zip directly from Kaggle's 'Output' panel on the right sidebar!")